Mapping/Merging

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("/content/tweets_with_sentiement_FIN.csv", low_memory=False)
mapping = pd.read_excel("/content/Brand by AdLength.xlsx")

metric_cols = [
    "public_metrics.like_count",
    "public_metrics.retweet_count",
    "public_metrics.reply_count",
    "public_metrics.quote_count",
]
for c in metric_cols:
    if c not in df.columns:
        df[c] = 0
df["total_engagement"] = df[metric_cols].fillna(0).sum(axis=1)

brand_perf = df.groupby("brand").agg(
    tweet_volume=("id", "count"),
    total_engagement=("total_engagement", "sum"),
    mean_engagement=("total_engagement", "mean"),
).reset_index()

brand_perf["engagement_per_tweet"] = brand_perf["total_engagement"] / brand_perf["tweet_volume"]

brand_perf["brand_join"] = brand_perf["brand"].str.replace(r"_\d+$", "", regex=True).str.strip()

mapping["brand_join"] = mapping["Brand"].astype(str).str.strip()

print("brand_perf rows:", brand_perf.shape)
print("mapping rows:", mapping.shape)


Visuals

In [ ]:
import matplotlib.ticker as ticker

plt.figure()

plt.scatter(merged["Ad_Length"], merged["total_engagement"])

plt.gca().yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, _: f"{int(x):,}")
)

plt.xlabel("Ad Length (seconds)")
plt.ylabel("Total Engagement")
plt.title("Ad Length vs Total Engagement")

plt.tight_layout()
plt.show()


In [ ]:
plt.figure()

plt.scatter(merged["Ad_Length"], merged["total_engagement"] / 1_000_000)

plt.xlabel("Ad Length (seconds)")
plt.ylabel("Total Engagement (Millions)")
plt.title("Ad Length vs Total Engagement")

plt.tight_layout()
plt.show()


In [ ]:
plt.figure()
plt.scatter(merged["Ad_Length"], merged["engagement_per_tweet"])
plt.xlabel("Ad Length (seconds)")
plt.ylabel("Engagement per Tweet")
plt.title("Ad Length vs Engagement Efficiency (Brand-level)")
plt.tight_layout()
plt.show()


In [ ]:
length_summary = merged.groupby("Ad_Length").agg(
    brands=("brand_join", "nunique"),
    total_tweets=("tweet_volume", "sum"),
    total_engagement=("total_engagement", "sum"),
    avg_engagement_per_tweet=("engagement_per_tweet", "mean"),
).reset_index().sort_values("Ad_Length")

plt.figure()
plt.bar(length_summary["Ad_Length"].astype(str), length_summary["avg_engagement_per_tweet"])
plt.xlabel("Ad Length (seconds)")
plt.ylabel("Avg Engagement per Tweet")
plt.title("Avg Engagement Efficiency by Ad Length")
plt.tight_layout()
plt.show()

length_summary
